# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

# Add uv to PATH for this notebook session
import os
os.environ["PATH"] += ":/home/pmanne/.local/bin"

# Verify it works
!which uv
!uv --version

# Create virtual environment
!uv venv .venv --seed --clear

# Install dependencies
!.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done.")

downloading uv 0.11.8 x86_64-unknown-linux-gnu
installing to /home/pmanne/.local/bin
  uv
  uvx
everything's installed!
/home/pmanne/.local/bin/uv
uv 0.11.8 (x86_64-unknown-linux-gnu)
Using CPython 3.11.9 interpreter at: /opt/conda/bin/python3
Creating virtual environment with seed packages at: .venv
 + packaging==26.2
 + pip==26.1
 + setuptools==82.0.1
 + wheel==0.47.0
Activate with: source .venv/bin/activate
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached transformers-5.6.2-py3-none-any.whl.metadata (33 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached antlr4_python3_runtime-4.11.1-py3-none-any.whl.metadata (291 bytes)
  Using cached ipykernel-7.2.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached jupyter-1.1.1-py2.py3-none-any.whl.meta

### Run the cell below every time to activate the installed environment. 

In [1]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [1]:
import json
import os
os.environ["BNB_CUDA_VERSION"] = "121"

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional
import multiprocessing
multiprocessing.set_start_method("spawn", force=True)

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm


INFO 05-24 10:30:54 [__init__.py:239] Automatically detected platform cuda.


In [2]:
import torch

# Returns True if a CUDA-compatible GPU is found and usable
print(torch.cuda.is_available())


True


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [3]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [4]:
SYSTEM_PROMPT_MATH = (
   "You are an expert mathematician. Solve the problem step-by-step."
   "Before giving the final answer, verify it using a different method or quick check."
   "Put your final answer inside \\boxed{}. "
   "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
   "e.g. \\boxed{3, 7}."
)


# SYSTEM_PROMPT_MCQ = (
#    "You are an expert mathematician."
#    "Read the problem and the answer choices below, then select the single best answer. "
#    "Do not guess. Solve the problem independently first, then compare your result to the answer choices."
#    "Maintain a list of known values and derived results, and update it after each step. Reuse them consistently."
#    "Before giving the final answer, verify it using a different method or quick check."
#    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
# )

SYSTEM_PROMPT_MCQ = (
   "You are an expert mathematician."
   "Read the problem and the answer choices below, then select the single best answer. "
   "Do not guess. Solve the problem independently first, then compare your result to the answer choices."
   "Maintain a list of known values and derived results, and update it after each step. Reuse them consistently."
   "Before giving the final answer, verify it using a different method or quick check."
   "Your FINAL output must be EXACTLY one boxed capital letter and nothing else, "
   "e.g. \\boxed{C}. "
   "Do not output explanations, reasoning, or extra text."
)



def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



In [5]:
import random

SUBSET_SIZE = 200
third_size = SUBSET_SIZE // 3

finetune_data_source = "data/public.jsonl"
data = [json.loads(line) for line in open(finetune_data_source)]
mcq = [d for d in data if d.get("options")]
frq = [d for d in data if not d.get("options")]

sampled_mcq = random.sample(mcq, third_size)
sampled_frq = random.sample(frq, third_size*2)

combined_subset = sampled_mcq + sampled_frq
random.shuffle(combined_subset)

output_filename = "data/finetuning_data.jsonl"

with open(output_filename, "w", encoding="utf-8") as f:
    for item in combined_subset:
        f.write(json.dumps(item) + "\n")

print(f"Successfully saved {len(combined_subset)} elements to {output_filename} \
        with {third_size} MCQ and {third_size*2} FRQ")


Successfully saved 198 elements to data/finetuning_data.jsonl         with 66 MCQ and 132 FRQ


In [6]:
reasoning_rows = []

for row in combined_subset:

    answer = row["answer"]

    if isinstance(answer, list):
        answer = ", ".join(map(str, answer))

    text = f"""
Question:
{row["question"]}

Solution:
Solve the problem step-by-step carefully.
Verify the arithmetic and reasoning.

Final Answer:
\\boxed{{{answer}}}
"""

    reasoning_rows.append({
        "text": text
    })

with open("data/finetuning_reasoning.jsonl", "w") as f:
    for row in reasoning_rows:
        f.write(json.dumps(row) + "\n")

print("Saved reasoning-format dataset.")

Saved reasoning-format dataset.


In [6]:
import sys
print(sys.executable)

/home/pmanne/.venv/bin/python


In [7]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.6.0+cu124
12.4
True
NVIDIA A30


In [8]:
import os
print(os.environ.get("BNB_CUDA_VERSION"))

121


## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token


llm = LLM(
    # model=MODEL_ID,
    # quantization="bitsandbytes",
    # load_format="bitsandbytes",
    # enable_prefix_caching=False,
    # gpu_memory_utilization=0.50,
    # max_model_len=16384,
    # trust_remote_code=True,
    # max_num_seqs=256,
    # max_num_batched_tokens=32768,
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    dtype="half",
    gpu_memory_utilization=0.50,
    max_model_len=8192,
    trust_remote_code=True,
)

# llm = LLM(
#     model=MODEL_ID,
#     tensor_parallel_size=1,
#     enforce_eager=True,
#     gpu_memory_utilization=0.50,
#     max_model_len=16384,
#     trust_remote_code=True,
#     max_num_seqs=64,
# )

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

# self_consistency_params = SamplingParams(
#     n=5,
#     max_tokens=MAX_TOKENS,
#     temperature=0.7,
#     top_p=0.95,
#     top_k=40,
#     repetition_penalty=1.0,
# )


# import re
# from collections import Counter

# def extract_boxed_answer(text):
#     matches = re.findall(r"\\boxed\{([^}]*)\}", text)
#     if not matches:
#         return None
#     return matches[-1].strip()

# def majority_vote(outputs):
#     answers = [extract_boxed_answer(o) for o in outputs]
#     answers = [a for a in answers if a is not None]

#     if not answers:
#         return None

#     return Counter(answers).most_common(1)[0][0]

print("Model loaded.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

INFO 05-24 10:32:08 [model_runner.py:1450] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utilization` or switching to eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.


Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 05-24 10:32:46 [model_runner.py:1592] Graph capturing finished in 38 secs, took 0.73 GiB


INFO 05-24 10:32:46 [llm_engine.py:437] init engine (profile, create kv cache, warmup model) took 50.05 seconds


Model loaded.


In [10]:
# %pip install unsloth datasets transformers trl 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/71.0 MB ? eta -:--:--

   ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/71.0 MB 7.9 MB/s eta 0:00:09

   ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/71.0 MB 6.5 MB/s eta 0:00:11

   ━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/71.0 MB 5.8 MB/s eta 0:00:12

   ━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/71.0 MB 5.3 MB/s eta 0:00:13

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━ 22/23 [unsloth]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━ 22/23 [unsloth]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━ 22/23 [unsloth]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23/23 [unsloth]


Note: you may need to restart the kernel to use updated packages.


In [8]:
import json

fixed_rows = []

with open("data/finetuning_data.jsonl") as f:
    for line in f:
        row = json.loads(line)

        answer = row["answer"]

        # Convert lists into a single string
        if isinstance(answer, list):
            answer = " | ".join(map(str, answer))

        # Convert everything into training text
        text = f"""Question:
{row["question"]}

Answer:
{answer}"""

        fixed_rows.append({
            "text": text
        })

with open("data/finetuning_data_fixed.jsonl", "w") as f:
    for row in fixed_rows:
        f.write(json.dumps(row) + "\n")

print("Saved cleaned dataset.")

Saved cleaned dataset.


In [9]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

max_seq_length = 1024
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

dataset = load_dataset("json", data_files="data/finetuning_reasoning.jsonl")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = True,
        bf16 = False,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

model.save_pretrained("qwen_math_lora")
tokenizer.save_pretrained("qwen_math_lora")


/home/pmanne/.venv/lib/python3.10/site-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *
/home/pmanne/.venv/lib/python3.10/site-packages/unsloth_zoo/__init__.py:404: UserWarning: Unsloth fused-forward install skipped: requires transformers >= 4.56.0.
  _install_fused_forward()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.5.5: Fast Qwen2 patching. Transformers: 4.51.3. vLLM: 0.8.5.post1.
   \\   /|    NVIDIA TITAN RTX. Num GPUs = 1. Max memory: 23.459 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

...

Generating train split: 0 examples [00:00, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/198 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 198 | Num Epochs = 3 | Total steps = 147
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss
10,2.207000
20,1.536000
30,1.167200
40,1.054800
50,1.057800
60,0.952000
70,0.894600
80,0.970400
90,0.901400
100,1.082400


('qwen_math_lora/tokenizer_config.json',
 'qwen_math_lora/special_tokens_map.json',
 'qwen_math_lora/vocab.json',
 'qwen_math_lora/merges.txt',
 'qwen_math_lora/added_tokens.json',
 'qwen_math_lora/tokenizer.json')

## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     torch_dtype=torch.float32,
#     device_map="cpu",
# )


/home/pmanne/cse151b_competition/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GeForce GTX 1080 Ti which is of cuda capability 6.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/home/pmanne/cse151b_competition/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/home/pmanne/cse151b_competition/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:435: UserWarning: 
NVIDIA GeForce GTX 1080 Ti with CUDA capability sm_61 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the NVIDIA GeForce GTX 1080 Ti GPU with PyTorch, please check the instructions

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Error named symbol not found at line 62 in file /src/csrc/ops.cu


In [2]:
import torch
print(torch.__version__)

2.10.0+cu128


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [10]:
from unsloth import FastLanguageModel
import torch

# Load your fine-tuned LoRA model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="qwen_math_lora",
    max_seq_length=2048,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

# Build prompts for first 5 entries
prompts = []

for item in data[:5]:

    system, user = build_prompt(
        item["question"],
        item.get("options")
    )

    prompt_text = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")

responses = []

for i, prompt_text in enumerate(prompts):

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    responses.append(response)

    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(response[:400], "..." if len(response) > 400 else "")

==((====))==  Unsloth 2026.5.5: Fast Qwen2 patching. Transformers: 4.51.3. vLLM: 0.8.5.post1.
   \\   /|    NVIDIA TITAN RTX. Num GPUs = 1. Max memory: 23.459 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ImportError: cannot import name 'ALL_PARALLEL_STYLES' from 'transformers.integrations.tensor_parallel' (/home/pmanne/.venv/lib/python3.10/site-packages/transformers/integrations/tensor_parallel.py)

In [6]:
# Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Generate
# print(f"Generating responses for {len(prompts)} questions...")
# outputs = llm.generate(prompts, sampling_params=sampling_params)

# responses = [out.outputs[0].text.strip() for out in outputs]

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")



from tqdm import tqdm
import json

# Recreate subset
data_subset = data[1:471]

# Resume from ID 184
remaining_data = data_subset[379:]

with open("predictions.jsonl", "a") as f:

    for idx, item in enumerate(tqdm(remaining_data), start=184):

        print(f"[{idx}/470] Generating... ID={item.get('id')}")

        system, user = build_prompt(
            item["question"],
            item.get("options")
        )

        prompt_text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )

        output = llm.generate(
            [prompt_text],
            sampling_params=sampling_params,
        )

        response = output[0].outputs[0].text.strip()

        result = {
            "id": item.get("id"),
            "response": response
        }

        f.write(json.dumps(result) + "\n")
        f.flush()

        print(f"✓ Saved response {idx}")


  0%|          | 0/91 [00:00<?, ?it/s]

[184/470] Generating... ID=380


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  1%|          | 1/91 [03:21<5:02:30, 201.67s/it]

...

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

100%|██████████| 91/91 [1:46:43<00:00, 53.86s/it]

100%|██████████| 91/91 [1:46:43<00:00, 70.37s/it]

✓ Saved response 274


In [7]:
import json

# Load existing predictions
rows = []
data_subset = data[1:471]
with open("predictions.jsonl", "r") as f:
    for line in f:
        rows.append(json.loads(line))

# Build lookup from original dataset
id_to_mcq = {}

for item in data_subset:
    is_mcq = item.get("options") is not None
    id_to_mcq[item.get("id")] = is_mcq

# Add new column
for row in rows:
    row["is_mcq"] = id_to_mcq.get(row["id"], False)

# Save updated file
with open("predictions_updated.jsonl", "w") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")

print("Done!")

Done!


In [9]:
# See first item
print(data_subset[0])

{'question': '$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $', 'options': ['$0$', '$frac{1}{a}$', '$frac{3}{a}$', '$frac{1}{2a^2}$', '$frac{1}{2a}$', '$frac{2}{a}$', '$2a$', '$frac{3}{2a}$', '$frac{3}{2a^2}$', '$frac{1}{a^2}$'], 'answer': 'F', 'id': 1}


### Generate with Transformers (for Datahub)

In [ ]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Tokenize (padded batch)
# print(f"Generating responses for {len(prompts)} questions...")
# inputs = tokenizer(
#     prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=16384,
# ).to(llm.device)

# # Generate
# with torch.no_grad():
#     output_ids = llm.generate(
#         **inputs,
#         max_new_tokens=MAX_TOKENS,
#         temperature=0.6,
#         top_p=0.95,
#         top_k=20,
#         repetition_penalty=1.0,
#         do_sample=True,
#     )

# # Decode only the new tokens (strip the prompt)
# responses = []
# for i, out in enumerate(output_ids):
#     new_tokens = out[inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

MAX_TOKENS = 512

item = data[0]
system, user = build_prompt(item["question"], item.get("options"))

prompt = tokenizer.apply_chat_template(
    [{"role": "system", "content": system},
     {"role": "user", "content": user}],
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=3000,
).to(llm.device)

with torch.no_grad():
    output = llm.generate(
        **inputs,
        max_new_tokens=MAX_TOKENS,
        do_sample=False,
    )

# Decode only generated tokens (not the original prompt)
new_tokens = output[0][inputs["input_ids"].shape[1]:]
response_text = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

# Store in responses list so scoring code works
responses = [response_text]

print(responses[0])

In [ ]:
!nvidia-sim

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [23]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 0/1126 [00:00<?, ?it/s]

Scoring:   0%|          | 5/1126 [00:00<00:22, 50.70it/s]

Scoring complete. 5 results.


## 8. Summary

Print accuracy broken down by question type.

In [24]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    1 /    2  (50.00%)
  Free-form  :    1 /    3  (33.33%)
  Overall    :    2 /    5  (40.00%)


In [25]:
for i in range(len(results)):
    print("=" * 80)
    print("QUESTION:")
    print(data[i]["question"])

    print("\nGOLD:")
    print(data[i]["answer"])

    print("\nMODEL RESPONSE:")
    print(results[i]["response"])

    print("\nCORRECT:")
    print(results[i]["correct"])

QUESTION:
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]

GOLD:
['325*(1+325)']

MODEL RESPONSE:
system
You are an expert mathematician. Solve the problem step-by-step.Before giving the final answer, verify it using a different method or quick check.Put your final answer inside \boxed{}. If the problem has multiple sub-answers, separate them by commas inside a single \boxed{}, e.g. \boxed{3, 7}.
user
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]
assistant
We start by identifying the sequence in question as the first 325 terms of the arithmetic sequence where each term is an even number.

An arithmetic sequence with the first term \(a_1\) and common difference \(d\) can be represented as:
\[ a_n = a_1 + (n-1)d \]

For our sequence, \(a_1 = 2\) (the first even number) and \(d = 2\) (each successive even number is 2 more than the previous one).

The formula for the sum of the first \(n\) terms (\(S_n\)) of an arithmetic sequence is given 

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

In [2]:
%pip install pandas
import pandas as pd

# Load your JSON data (from a file)
df = pd.read_json('predictions_updated.jsonl', lines=True)

# Alternatively, if you have a JSON string or list of dicts:
# df = pd.DataFrame(data_variable)

# Export to CSV
df.to_csv('output_file1.csv', index=False)

Note: you may need to restart the kernel to use updated packages.


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!